## ¡Báñate con dinero!

In [88]:
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
import json
import requests
from functools import reduce
import numpy as np
path = "../datos"
#path = "/content"

In [89]:
df = pd.read_excel(path + "/6N.1.1.C_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-3] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
#df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[:, [1, 2]]
df.columns = ["Entidad", "Agua y Drenaje"]
df['Agua y Drenaje'] = pd.to_numeric(df['Agua y Drenaje'], errors="coerce")
last_3 = df.nsmallest(3, 'Agua y Drenaje', keep='all')
top_1 = df.nlargest(1, 'Agua y Drenaje', keep='all')
max_val = top_1.values.tolist()[0][1]
nacional_val = df.loc[df['Entidad'] == "Estados Unidos Mexicanos"].values.tolist()[0][1]
prop_aguaYsaneam = df.loc[df['Entidad'] == "Baja California Sur"].values.tolist()[0][1]

fig = go.Figure()
df=last_3
fig.add_trace(go.Bar(
    x=df['Entidad'],
    y=df['Agua y Drenaje'],
    marker_color='#64bec0',
    name='Entidad',
    hovertemplate=(
        "<b>Estado:</b> %{x}<br>" +
        "<b>Proporción:</b> %{y:.2f}%<br>" +
        "<extra></extra>"
    )
))

# 3. Agregar la Línea Nacional (Horizontal)
fig.add_hline(
    y=nacional_val,
    line_dash="dash", # Línea punteada
    line_color="red",
    line_width=3,
    annotation_text=f"Promedio Nacional: {nacional_val:.2f} " + "%",
    annotation_position="top right",
    annotation_font=dict(color="red", size=14)
)
# fig.add_hline(
#     y=max_val,
#     line_dash="dot", # Línea punteada
#     line_color="#b61420",
#     line_width=3,
#     annotation_text=f"1° lugar Chihuahua: {max_val:.2f} " + "%",
#     annotation_position="top right",
#     annotation_font=dict(color="#b61420", size=14)
# )

# 4. Estética del Layout
fig.update_layout(
    title="% Acceso a agua y drenaje <br> <sup>CONAGUA 2022</sup>",
    xaxis_title="Los 3 estados con el menor acceso",
    yaxis_title="Porcentaje (%)",
    plot_bgcolor='white',
    height=500
)

# Añadir rejilla suave en el eje Y para mejor lectura
fig.update_yaxes(showgrid=True, gridcolor='lightgray')

fig.show()

# Venn

In [90]:
# Pob agua
df = pd.read_excel(path + "/6.1.1.C_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-7] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
#df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[:, [1, 2]]
df.columns = ["Entidad", "Agua"]
df['Agua'] = pd.to_numeric(df['Agua'], errors="coerce")
df_agua = df

# Pob Saneamiento
df = pd.read_excel(path + "/6.2.1.C_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-3] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
#df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[:, [1, 2]]
df.columns = ["Entidad", "Saneamiento"]
df['Saneamiento'] = pd.to_numeric(df['Saneamiento'], errors="coerce")
df_saneam = df

prop_agua = df_agua.loc[df['Entidad'] == "Baja California Sur"].values.tolist()[0][1]
prop_saneam = df_saneam.loc[df['Entidad'] == "Baja California Sur"].values.tolist()[0][1]

# Probas calculadas
data = {
    "Solo agua": prop_agua - prop_aguaYsaneam,
    "Solo saneamiento": prop_saneam - prop_aguaYsaneam,
    "Agua y Saneamiento": prop_aguaYsaneam,
    "Ni agua, Ni saneamiento": 100 - (prop_agua + prop_saneam - prop_aguaYsaneam)
}

fig = go.Figure()

# Crear el círculo para el conjunto A
fig.add_shape(type="circle",
    line_color="white", fillcolor="Blue", opacity=0.3,
    x0=0, y0=0, x1=1.5, y1=1.5
)

# Crear el círculo para el conjunto B (desplazado a la derecha)
fig.add_shape(type="circle",
    line_color="white", fillcolor="Gray", opacity=0.3,
    x0=1, y0=0, x1=3, y1=2
)

# Añadir etiquetas de texto
fig.add_trace(go.Scatter(
    x=[0.5, 2.3, 1.25, 1.5],
    y=[0.7, 1, 1, 2.2],
    text=[f"Solo agua<br>{data['Solo agua']:.2f}%",
          f"Solo saneamiento<br>{data['Solo saneamiento']:.2f}%",
          f"Ambos<br>{data['Agua y Saneamiento']:.2f}%",
          f"Ni agua, Ni saneamiento: {data['Ni agua, Ni saneamiento']:.2f}%"],
    mode="text",
    textfont=dict(size=14, color="black")
))

# Configuración del diseño
fig.update_layout(
    title="Distribución de la población en BCS,<br>respecto de su acceso a los servicios<br><sup> CONAGUA 2022</sup>",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 3.5]),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, 2.5]),
    plot_bgcolor="white",
    width=600, height=500
)

fig.show()

## No se reutilizan las aguas residuales

In [91]:
from plotly.colors import colorbrewer
df = pd.read_excel(path + "/Unidades-economicas-grandes-que-realizan-tratamiento-de-aguas-residuales.xlsx", sheet_name=0, header = None)
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[:, [0,1, 2]]
df.columns = ["Valores", "Porcentaje", "Categoría"]
df["Valores"] = pd.to_numeric(df["Valores"], errors='coerce')
df["Porcentaje"] = pd.to_numeric(df["Porcentaje"], errors='coerce')

df["Categoría"] = [
    "Sectores que SÍ trataron aguas residuales",
    "Sectores que NO trataron aguas residuales"
]
colores={"Sectores que SÍ trataron aguas residuales":	'#552222',
    "Sectores que NO trataron aguas residuales": '#607435' }

fig = px.pie(
    df,
    values='Porcentaje',
    names='Categoría',
    title='Tratamiento de aguas residuales por unidades económicas grandes en BCS 2019<br><sup> INEGI, Censos económicos 2019',
    hole=0.5,  # <--- Esto lo convierte en dona (0.5 = 50% de radio)
    color='Categoría',
    color_discrete_map= colores
)

fig.show()